In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
from openmmtools.integrators import LangevinIntegrator, LangevinSplittingGirsanov
import matplotlib

import numpy as np
from matplotlib import pyplot as plt

import gc

from scipy.interpolate import CubicSpline
from potential import *

In [ ]:
##############################################################################
# Global parameters
##############################################################################
mass = 1.0 * dalton
temp = 298.15
temperature = temp * kelvin
collision_rate = 10 / picosecond
timestep = 5 * femtosecond
splitting = 'R V O V R'
nstxout = 20
kt = temp*8.314/1000
platform = Platform.getPlatformByName('CUDA')

In [ ]:
# matplotlib fonts
# matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 20

In [ ]:
fig,ax = plt.subplots(figsize=(10,2.5))
DoubleWellForce().plot(ax=ax,linewidth=3,color='black')
ax.tick_params(bottom=False, labelbottom=False)
ax.set_ylim(0,30)
ax.set_xlim(-1,1)
ax.set_xlabel('$x$')
ax.set_ylabel('Energy (kJ/mol)')
plt.savefig('figures/1d_doublewell/energy.png',dpi=600)

In [ ]:
fig,ax = plt.subplots(figsize=(12,2.5))
QuadraWellForce().plot(ax=ax,linewidth=3,color='black')
ax.tick_params(bottom=False, labelbottom=False)
ax.set_ylim(0,30)
ax.set_xlim(-1,1)
ax.set_xlabel('$x$')
ax.set_ylabel('Energy (kJ/mol)')
plt.savefig('figures/1d_quadrawell/energy.png',dpi=600)

#### 0. Unbiased Simulation as Reference

##### 0.1 1D doublewell potential

In [ ]:
# Simulation setup
n_sim = 10
n_iters = 1000000

# Run 1D doublewell simulation
system = System()
systemforce = DoubleWellForce()
starting_positions = np.concatenate(
    [np.tile([-0.5,0,0],(int(n_sim/2),1)),np.tile([0.5,0,0],(int(n_sim/2),1))],axis=0
)

# For unbiased simulation, we can parallelize the independent runs.
for n in range(n_sim):
    system.addParticle(mass)
    systemforce.addParticle(n, [])
system.addForce(systemforce)

topology = app.Topology()

# Langevin integrator that allows girsanov reweighting factor output
integrator = LangevinSplittingGirsanov(
    nstxout = nstxout, temperature = temperature,collision_rate = collision_rate,
    timestep = timestep, splitting = splitting
)

# Create simulation
simulation = Simulation(topology,system,integrator,platform)
simulation.context.setPositions(starting_positions)
simulation.context.setVelocitiesToTemperature(temperature)

In [ ]:
%%time
save_dir = 'traj_and_dat/1d_doublewell/unbiased_{n_sim}x{n_iters}.npy'.format(
    n_sim = n_sim, n_iters = n_iters
)

data = np.zeros((n_sim,n_iters))
for j in range(n_iters):
    if j % 1000 == 0:
        print(str(j)+'/'+str(n_iters))
    data[:,j] = simulation.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(nanometer)[:,0]
    simulation.step(nstxout)

np.save(save_dir,data)

##### 0.2 1D Quadrawell Potential

In [ ]:
# Simulation setup
n_sim = 10
n_iters = 1000000

# Run 1D quadrawell simulation
system = System()
systemforce = QuadraWellForce()
starting_positions = np.concatenate(
    [np.tile([-0.75,0,0],(int(n_sim/2),1)),
     np.tile([0.75,0,0],(int(n_sim/2),1)),],axis=0
)

# For unbiased simulation, we can parallelize the independent runs.
for n in range(n_sim):
    system.addParticle(mass)
    systemforce.addParticle(n, [])
system.addForce(systemforce)

topology = app.Topology()

# Langevin integrator that allows girsanov reweighting factor output
integrator = LangevinSplittingGirsanov(
    nstxout = nstxout, temperature = temperature,collision_rate = collision_rate,
    timestep = timestep, splitting = splitting
)

# Create simulation
simulation = Simulation(topology,system,integrator,platform)
simulation.context.setPositions(starting_positions)
simulation.context.setVelocitiesToTemperature(temperature)

In [ ]:
%%time
save_dir = 'traj_and_dat/1d_quadrawell/unbiased_{n_sim}x{n_iters}.npy'.format(
    n_sim = n_sim, n_iters = n_iters
)

data = np.zeros((n_sim,n_iters))
for j in range(n_iters):
    if j % 1000 == 0:
        print(str(j)+'/'+str(n_iters))
    data[:,j] = simulation.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(nanometer)[:,0]
    simulation.step(nstxout)

np.save(save_dir,data)

#### 1. Umbrella Sampling

##### 1.1 1d doublewell potential

In [ ]:
### Umbrella Sampling: create windows
# Harmonic potential as in the conventional umbrella sampling: V(x) = k*(x-x0)^2
# Spring constant
k_spring = 100
# Window centers
n_windows = 50
x_min = -0.8
x_max = 0.8
window_centers = np.linspace(x_min,x_max,n_windows)
'''
# Visualize the windows
# Harmonic potential
def harmonic_bias(x,x0,k):
    return 0.5*k*(x-x0)**2
# Conventional umbrella sampling bias potential
fig,ax = plt.subplots()

DoubleWellForce().plot(ax=ax,linewidth=3,color='black')
window_range = (x_max-x_min) / n_windows + 0.35
x = np.linspace(-1,1,250)

for x0_i in window_centers:
    xw1 = np.linspace(x0_i-window_range,x0_i+window_range,100)
    bias_i = harmonic_bias(xw1,x0_i,k_spring)
    ax.plot(xw1,bias_i)
'''

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters_total = 100000
n_iters = int(n_iters_total / n_windows)

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 3), 3 for x, boltzmann factor, girsanov factor
    save_dir = 'traj_and_dat/1d_doublewell/umbrella_sampling/{n_windows}x{n_iters}_k={k_spring}_umbrella_sampling_{n_re}.npy'.format(
        k_spring = k_spring, n_windows = n_windows, n_iters = n_iters, n_re = n_re
    )
    data = np.zeros((len(window_centers),n_iters,3))
    for i,x0_i in enumerate(window_centers):
        print('Running the {i}/{n_windows} Window of Umbrella Sampling Simulation ...'.format(
            i = i, n_windows = n_windows
        ))
        # starting US simulation at current window center
        starting_position = np.array([[x0_i,0,0]])
        ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
        ### using context.reinitialize()
        system = System()
        systemforce = DoubleWellForce()
        system.addParticle(mass)
        systemforce.addParticle(0, [])
        system.addForce(systemforce)

        # Current window
        bias = CustomExternalForce('0.5*{k}*(x-{x0})^2'.format(k=k_spring,x0=x0_i))
        bias.setForceGroup(1)
        bias.addParticle(0,[])
        system.addForce(bias)

        topology = app.Topology()
        
        # Langevin integrator that allows girsanov reweighting factor output
        integrator = LangevinSplittingGirsanov(
            nstxout = nstxout, temperature = temperature,
            collision_rate = collision_rate, timestep = timestep,
            splitting = splitting
        )

        # Create simulation
        simulation = Simulation(topology,system,integrator,platform)
        simulation.context.setPositions(starting_position)
        simulation.context.setVelocitiesToTemperature(temperature)
        
        for k in range(n_iters):
            if k % 100 == 0:
                print(str(k)+'/'+str(n_iters))
            # Record Positions
            data[i,k,0] = simulation.context.getState(
                getPositions=True
            ).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
            # Record Boltzmann weight
            bias_potential = simulation.context.getState(
                getEnergy=True,groups=0b00000000000000000000000000000010
            ).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
            data[i,k,1] = np.exp(bias_potential / kt)
            # Step simulation
            simulation.step(nstxout)
            # Record Girsanov weight
            data[i,k,2] = simulation.integrator.getGlobalVariableByName("M")
    np.save(save_dir,data)

##### 1.2 1d quadrawell potential

In [ ]:
### Umbrella Sampling: create windows
# Harmonic potential as in the conventional umbrella sampling: V(x) = k*(x-x0)^2
# Spring constant
k_spring = 100
# Window centers
n_windows = 50
x_min = -0.8
x_max = 0.8
window_centers = np.linspace(x_min,x_max,n_windows)
'''
# Visualize the windows
# Harmonic potential
def harmonic_bias(x,x0,k):
    return 0.5*k*(x-x0)**2
# Conventional umbrella sampling bias potential
fig,ax = plt.subplots()

QuadraWellForce().plot(ax=ax,linewidth=3,color='black')
window_range = (x_max-x_min) / n_windows + 0.35
x = np.linspace(-1,1,250)

for x0_i in window_centers:
    xw1 = np.linspace(x0_i-window_range,x0_i+window_range,100)
    bias_i = harmonic_bias(xw1,x0_i,k_spring)
    ax.plot(xw1,bias_i)
'''

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters_total = 100000
n_iters = int(n_iters_total / n_windows)

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 3), 3 for x, boltzmann factor, girsanov factor
    save_dir = 'traj_and_dat/1d_quadrawell/umbrella_sampling/{n_windows}x{n_iters}_k={k_spring}_umbrella_sampling_{n_re}.npy'.format(
        k_spring = k_spring, n_windows = n_windows, n_iters = n_iters, n_re = n_re
    )
    data = np.zeros((len(window_centers),n_iters,3))
    for i,x0_i in enumerate(window_centers):
        print('Running the {i}/{n_windows} Window of Umbrella Sampling Simulation ...'.format(
            i = i, n_windows = n_windows
        ))
        # starting US simulation at current window center
        starting_position = np.array([[x0_i,0,0]])
        ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
        ### using context.reinitialize()
        system = System()
        systemforce = QuadraWellForce()
        system.addParticle(mass)
        systemforce.addParticle(0, [])
        system.addForce(systemforce)

        # Current window
        bias = CustomExternalForce('0.5*{k}*(x-{x0})^2'.format(k=k_spring,x0=x0_i))
        bias.setForceGroup(1)
        bias.addParticle(0,[])
        system.addForce(bias)

        topology = app.Topology()
        
        # Langevin integrator that allows girsanov reweighting factor output
        integrator = LangevinSplittingGirsanov(
            nstxout = nstxout, temperature = temperature,
            collision_rate = collision_rate, timestep = timestep,
            splitting = splitting
        )

        # Create simulation
        simulation = Simulation(topology,system,integrator,platform)
        simulation.context.setPositions(starting_position)
        simulation.context.setVelocitiesToTemperature(temperature)
        
        for k in range(n_iters):
            if k % 100 == 0:
                print(str(k)+'/'+str(n_iters))
            # Record Positions
            data[i,k,0] = simulation.context.getState(
                getPositions=True
            ).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
            # Record Boltzmann weight
            bias_potential = simulation.context.getState(
                getEnergy=True,groups=0b00000000000000000000000000000010
            ).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
            data[i,k,1] = np.exp(bias_potential / kt)
            # Step simulation
            simulation.step(nstxout)
            # Record Girsanov weight
            data[i,k,2] = simulation.integrator.getGlobalVariableByName("M")
    np.save(save_dir,data)

#### 2. Metadynamics Rerun

##### 2.1 1d doublewell potential

In [ ]:
biasfactor = 2

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters = 100000

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 3), 3 for x, boltzmann factor, girsanov factor
    save_dir = 'traj_and_dat/1d_doublewell/metad_rerun/{n_iters}_gamma={biasfactor}_metad_rerun_{n_re}.npy'.format(
        biasfactor = biasfactor, n_iters = n_iters, n_re = n_re
    )
    data = np.zeros((n_iters,3))
    
    # starting metad rerun simulation at the left basin
    starting_position = np.array([[-0.5,0,0]])
    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = DoubleWellForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Converged metadynamics bias
    bias = CustomExternalForce('(1/{biasfactor}-1)*'.format(biasfactor=biasfactor)+systemforce.expression)
    bias.setForceGroup(1)
    bias.addParticle(0,[])
    system.addForce(bias)

    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )

    # Create simulation
    simulation = Simulation(topology,system,integrator,platform)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    
    for k in range(n_iters):
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
        # Record Positions
        data[k,0] = simulation.context.getState(
            getPositions=True
        ).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
        # Record Boltzmann weight
        bias_potential = simulation.context.getState(
            getEnergy=True,groups=0b00000000000000000000000000000010
        ).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
        data[k,1] = np.exp(bias_potential / kt)
        # Step simulation
        simulation.step(nstxout)
        # Record Girsanov weight
        data[k,2] = simulation.integrator.getGlobalVariableByName("M")
    np.save(save_dir,data)

##### 2.2 1d quadrawell potential

In [ ]:
biasfactor = 2

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters = 100000

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 3), 3 for x, boltzmann factor, girsanov factor
    save_dir = 'traj_and_dat/1d_quadrawell/metad_rerun/{n_iters}_gamma={biasfactor}_metad_rerun_{n_re}.npy'.format(
        biasfactor = biasfactor, n_iters = n_iters, n_re = n_re
    )
    data = np.zeros((n_iters,3))
    
    # starting metad rerun simulation at the left basin
    starting_position = np.array([[-0.5,0,0]])
    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = QuadraWellForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Converged metadynamics bias
    bias = CustomExternalForce('(1/{biasfactor}-1)*'.format(biasfactor=biasfactor)+systemforce.expression)
    bias.setForceGroup(1)
    bias.addParticle(0,[])
    system.addForce(bias)

    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )

    # Create simulation
    simulation = Simulation(topology,system,integrator,platform)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    
    for k in range(n_iters):
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
        # Record Positions
        data[k,0] = simulation.context.getState(
            getPositions=True
        ).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
        # Record Boltzmann weight
        bias_potential = simulation.context.getState(
            getEnergy=True,groups=0b00000000000000000000000000000010
        ).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
        data[k,1] = np.exp(bias_potential / kt)
        # Step simulation
        simulation.step(nstxout)
        # Record Girsanov weight
        data[k,2] = simulation.integrator.getGlobalVariableByName("M")
    np.save(save_dir,data)

##### 2.3 2d Muller potential

In [ ]:
biasfactor = 2

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters = 100000

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 4), 4 for x, y, boltzmann factor, girsanov factor
    save_dir = 'traj_and_dat/2d_muller/metad_rerun/{n_iters}_gamma={biasfactor}_metad_rerun_{n_re}.npy'.format(
        biasfactor = biasfactor, n_iters = n_iters, n_re = n_re
    )
    data = np.zeros((n_iters,4))
    
    # starting metad rerun simulation at the top-left basin
    starting_position = np.array([[-0.57,1.43,0]])
    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = MullerForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Converged metadynamics bias
    bias = CustomExternalForce('(1/{biasfactor}-1)*'.format(biasfactor=biasfactor)+systemforce.expression)
    bias.setForceGroup(1)
    bias.addParticle(0,[])
    system.addForce(bias)

    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )

    # Create simulation
    simulation = Simulation(topology,system,integrator,platform)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    
    for k in range(n_iters):
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
        # Record Positions
        data[k,:2] = simulation.context.getState(
            getPositions=True
        ).getPositions(asNumpy=True).value_in_unit(nanometer)[0,:2]
        # Record Boltzmann weight
        bias_potential = simulation.context.getState(
            getEnergy=True,groups=0b00000000000000000000000000000010
        ).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
        data[k,2] = np.exp(bias_potential / kt)
        # Step simulation
        simulation.step(nstxout)
        # Record Girsanov weight
        data[k,3] = simulation.integrator.getGlobalVariableByName("M")
    np.save(save_dir,data)

#### 3. Metadynamics build-up simulation

##### 3.1 1d doublewell potential

In [ ]:
# Metadynamics parameters
biasfactor = 2
cv_sigma = 0.1
periodic_cv = False
gridwidth = 100
height = 1.2 * kilojoule_per_mole
pace = 100

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters = 100000

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 3), 3 for x, boltzmann factor, girsanov factor
    save_dir = 'traj_and_dat/1d_doublewell/metad_build_up/{n_iters}_gamma={biasfactor}_metad_build_up_{n_re}.npy'.format(
        biasfactor = biasfactor, n_iters = n_iters, n_re = n_re
    )
    data = np.zeros((n_iters,3))
    
    # starting metad rerun simulation at the left basin
    starting_position = np.array([[-0.5,0,0]])
    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = DoubleWellForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Define the collective variable as x
    cv_force = CustomExternalForce('x')
    cv_force.addParticle(0, [])
    
    # Create the BiasVariable and Metadynamics objects
    cv = BiasVariable(
        cv_force, systemforce.x_range[0], systemforce.x_range[1], cv_sigma, periodic_cv, gridwidth
    )
        
    metad = Metadynamics(
        system, [cv], temperature, biasfactor, height, pace#, save_freq, save_dir
    )
    
    # Set ForceGroup to 1 for Girsanov path weights
    metad._force.setForceGroup(1)

    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )

    # Create simulation
    simulation = Simulation(topology,system,integrator,platform)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    
    for k in range(n_iters):
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
        # Record Positions
        data[k,0] = simulation.context.getState(
            getPositions=True
        ).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
        # Step simulation
        metad.step(simulation,nstxout)
        # Record Girsanov weight
        data[k,2] = simulation.integrator.getGlobalVariableByName("M")

    # End-point bias potential reweighting: assuming pseudo-static bias
    # Retrieve the bias grid
    bias_grid = np.array(metad._table.getFunctionParameters()[0])
    cv_grid = np.linspace(systemforce.x_range[0],systemforce.x_range[1],bias_grid.shape[0])
    # Re-construct the cubic spline
    spline = CubicSpline(cv_grid,bias_grid,bc_type='natural')
    data[:,1] = np.exp(spline(data[:,0])/kt)
    
    np.save(save_dir,data)

##### 3.2 1d quadrawell potential

In [ ]:
# Metadynamics parameters
biasfactor = 2
cv_sigma = 0.1
periodic_cv = False
gridwidth = 100
height = 1.2 * kilojoule_per_mole
pace = 100

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters = 100000

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 3), 3 for x, boltzmann factor, girsanov factor
    save_dir = 'traj_and_dat/1d_quadrawell/metad_build_up/{n_iters}_gamma={biasfactor}_metad_build_up_{n_re}.npy'.format(
        biasfactor = biasfactor, n_iters = n_iters, n_re = n_re
    )
    data = np.zeros((n_iters,3))
    
    # starting metad rerun simulation at the left basin
    starting_position = np.array([[-0.5,0,0]])
    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = QuadraWellForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Define the collective variable as x
    cv_force = CustomExternalForce('x')
    cv_force.addParticle(0, [])
    
    # Create the BiasVariable and Metadynamics objects
    cv = BiasVariable(
        cv_force, systemforce.x_range[0], systemforce.x_range[1], cv_sigma, periodic_cv, gridwidth
    )
        
    metad = Metadynamics(
        system, [cv], temperature, biasfactor, height, pace#, save_freq, save_dir
    )
    
    # Set ForceGroup to 1 for Girsanov path weights
    metad._force.setForceGroup(1)

    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )

    # Create simulation
    simulation = Simulation(topology,system,integrator,platform)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    
    for k in range(n_iters):
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
        # Record Positions
        data[k,0] = simulation.context.getState(
            getPositions=True
        ).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
        # Step simulation
        metad.step(simulation,nstxout)
        # Record Girsanov weight
        data[k,2] = simulation.integrator.getGlobalVariableByName("M")

    # End-point bias potential reweighting: assuming pseudo-static bias
    # Retrieve the bias grid
    bias_grid = np.array(metad._table.getFunctionParameters()[0])
    cv_grid = np.linspace(systemforce.x_range[0],systemforce.x_range[1],bias_grid.shape[0])
    # Re-construct the cubic spline
    spline = CubicSpline(cv_grid,bias_grid,bc_type='natural')
    data[:,1] = np.exp(spline(data[:,0])/kt)
    
    np.save(save_dir,data)

#### 4. Steered MD

##### 4.1 1d doublewell potential

In [ ]:
# Spring constant
k_spring = 100

# Pulling from -0.8 to 0.8 with pulling_iters steps, 
# and then reverse the pulling direction
x_max = 0.8
x_min = -0.8
pulling_iters = 1000
v_pulling = (x_max - x_min) / pulling_iters              # cv value/iteration

In [ ]:
# starting the pulling simulation at one end
starting_position = np.array([[x_min,0,0]])
x0_update_sign = 1.0       # we first increase x0 from minx to maxx

# Pulling Simulation
n_repeats = 10
n_iters = 100000

for n_re in range(n_repeats):
    save_dir = 'traj_and_dat/1d_doublewell/steered_md/{n_iters}_v_pulling={v_pulling}_steered_md_{n_re}.npy'.format(
            v_pulling = v_pulling, n_iters = n_iters, n_re = n_re
        )
    data = np.zeros((n_iters,2))

    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = DoubleWellForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)
    
    # Initialize Pulling force
    pullingforce = CustomExternalForce('0.5 * {k_spring} * (x-x0)^2'.format(k_spring=k_spring))
    pullingforce.addParticle(0,[])
    pullingforce.addGlobalParameter('x0', starting_position[0][0])
    pullingforce.setForceGroup(1)
    system.addForce(pullingforce)
        
    # Create a dummy topology object
    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )
    
    # Create simulation
    simulation = Simulation(topology,system,integrator)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    for k in range(n_iters):
        x0_k = simulation.context.getParameter('x0')
        # Printing
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
            print('Current x0: {x0_k}'.format(x0_k=x0_k))
        # Update x0
        if x0_k >= x_max:
            x0_update_sign = -1.0    # reverse the sign so x0 starts to decrease
            print('x0 will decrease now.')
        elif x0_k <= x_min:
            x0_update_sign = 1.0     # reverse the sign so x0 starts to increase
            print('x0 will increase now.')
        else:
            None
        
        data[k,0] = simulation.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
        simulation.step(nstxout)
        data[k,1] = simulation.integrator.getGlobalVariableByName("M")
        
        x0_new = x0_k + x0_update_sign * v_pulling
        simulation.context.setParameter('x0',x0_new)

    np.save(save_dir,data)

##### 4.2 1d quadrawell potential

In [ ]:
# Spring constant
k_spring = 100

# Pulling from -0.8 to 0.8 with pulling_iters steps, 
# and then reverse the pulling direction
x_max = 0.8
x_min = -0.8
pulling_iters = 1000
v_pulling = (x_max - x_min) / pulling_iters              # cv value/iteration

In [ ]:
# starting the pulling simulation at one end
starting_position = np.array([[x_min,0,0]])
x0_update_sign = 1.0       # we first increase x0 from minx to maxx

# Pulling Simulation
n_repeats = 10
n_iters = 100000

for n_re in range(n_repeats):
    save_dir = 'traj_and_dat/1d_quadrawell/steered_md/{n_iters}_v_pulling={v_pulling}_steered_md_{n_re}.npy'.format(
            v_pulling = v_pulling, n_iters = n_iters, n_re = n_re
        )
    data = np.zeros((n_iters,2))

    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = QuadraWellForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)
    
    # Initialize Pulling force
    pullingforce = CustomExternalForce('0.5 * {k_spring} * (x-x0)^2'.format(k_spring=k_spring))
    pullingforce.addParticle(0,[])
    pullingforce.addGlobalParameter('x0', starting_position[0][0])
    pullingforce.setForceGroup(1)
    system.addForce(pullingforce)
        
    # Create a dummy topology object
    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )
    
    # Create simulation
    simulation = Simulation(topology,system,integrator)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    for k in range(n_iters):
        x0_k = simulation.context.getParameter('x0')
        # Printing
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
            print('Current x0: {x0_k}'.format(x0_k=x0_k))
        # Update x0
        if x0_k >= x_max:
            x0_update_sign = -1.0    # reverse the sign so x0 starts to decrease
            print('x0 will decrease now.')
        elif x0_k <= x_min:
            x0_update_sign = 1.0     # reverse the sign so x0 starts to increase
            print('x0 will increase now.')
        else:
            None
        
        data[k,0] = simulation.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(nanometer)[0][0]
        simulation.step(nstxout)
        data[k,1] = simulation.integrator.getGlobalVariableByName("M")
        
        x0_new = x0_k + x0_update_sign * v_pulling
        simulation.context.setParameter('x0',x0_new)

    np.save(save_dir,data)